In [4]:
from transformers import AutoTokenizer, AutoModel

slug = 'distilbert/distilbert-base-uncased'

In [5]:
import huggingface_hub as hfh
from datasets import load_dataset
hfh.login()

dsname = 'cornell-movie-review-data/rotten_tomatoes'

full_dataset = load_dataset(path=dsname, cache_dir='./mydatasets')
trainset = full_dataset['train']
validset = full_dataset['validation']


In [ ]:
from transformers import BertTokenizer
tokenizer = BertTokenizer()

def data_process(sample):
    return tokenizer(sample['text'], max_length=512, padding='max_length')

def prepare_dataset(ds):
    ds = ds.map(data_process)
    ds.rename_column('label', 'labels')
    ds.set_format('pt', columns=['input_ids', 'attention_mask', 'labels'], output_all_columns=True)
    return ds


trainset = trainset.map(data_process)
trainset.set_format('pt', columns=['input_ids'], output_all_columns=True)

validset = validset.map(data_process)
validset.set_format('pt', columns=['input_ids'], output_all_columns=True)

Map: 100%|██████████| 1066/1066 [00:00<00:00, 1847.37 examples/s]


In [25]:
from transformers import DistilBertForSequenceClassification
model = DistilBertForSequenceClassification.from_pretrained(slug, num_labels=2).cuda()
model

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 12364.92it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [26]:
import numpy as np
from sklearn.metrics import accuracy_score
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
from transformers import TrainerCallback

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

class TrainEvalCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        trainer = kwargs["trainer"]

        train_metrics = trainer.evaluate(
            eval_dataset=trainer.train_dataset,
            metric_key_prefix="train"
        )

        test_metrics = trainer.evaluate(
            eval_dataset=trainer.eval_dataset,
            metric_key_prefix="test"
        )

        print(train_metrics)
        print(test_metrics)

dc = DataCollatorWithPadding(tokenizer)

train_args = TrainingArguments(
    output_dir='./output',
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    logging_strategy='epoch',
    logging_steps=1,
    eval_strategy='epoch',
    learning_rate=5e-5
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=trainset,
    eval_dataset=validset,
    data_collator=dc,
    compute_metrics=compute_metrics
)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.696369,0.693196,0.500000
2,0.694226,0.693255,0.500000
3,0.693626,0.693399,0.500000
4,0.693105,0.693157,0.500000
5,0.693452,0.693229,0.500000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.71it/s]


TrainOutput(global_step=1335, training_loss=0.6941556094737535, metrics={'train_runtime': 539.0034, 'train_samples_per_second': 79.128, 'train_steps_per_second': 2.477, 'total_flos': 5649734552678400.0, 'train_loss': 0.6941556094737535, 'epoch': 5.0})

In [ ]:
np.unique(trainset['label'])
np.unique(validset['label'])